# Kaggle_notebook — Stable multi-model training pipeline
This notebook was used to train BERT, RoBERTa, and XLNet. I cleaned up extraneous notes and historical comments — no code or path changes were made.
Keep code cells intact for reproducibility.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -q transformers scikit-learn pandas numpy torch tqdm sentencepiece tensorboard

In [ ]:
import os
from pathlib import Path
import json, re, math, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, precision_recall_fscore_support, classification_report
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter
import warnings
warnings.filterwarnings('ignore')
# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
# Paths (works on Kaggle)
kaggle_base = Path('/kaggle/input')
found = list(kaggle_base.rglob('emotions.txt'))
if found:
    DATA_DIR = found[0].parent
else:
    DATA_DIR = Path('New folder')
print('DATA_DIR:', DATA_DIR)
TRAIN_PATH = DATA_DIR / 'train.tsv'
DEV_PATH = DATA_DIR / 'dev.tsv'
TEST_PATH = DATA_DIR / 'test.tsv'
EMOTIONS_PATH = DATA_DIR / 'emotions.txt'
OUT_DIR = Path('/kaggle/working') / 'train4_artifacts'
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUT_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)
METRICS_DIR = OUT_DIR / 'metrics'
METRICS_DIR.mkdir(exist_ok=True, parents=True)
TB_DIR = OUT_DIR / 'tb'
TB_DIR.mkdir(exist_ok=True, parents=True)
print('Out dir:', OUT_DIR)


In [ ]:
# Config - conservative for stability and speed
MODEL_NAME = 'bert-base-uncased' # {'roberta-base', 'distilbert-base-uncased', 'bert-base-uncased'}
NUM_EMOTIONS = 28
MAX_LEN = 128
BATCH_SIZE = 8
EVAL_BATCH = 16
NUM_EPOCHS = 6
BACKBONE_LR = 1e-5
HEAD_LR = 5e-5
WEIGHT_DECAY = 0.01
WARMUP = 200
MAX_GRAD_NORM = 1.0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device ->', DEVICE)


In [ ]:
# Data helpers
def read_tsv(path):
    return pd.read_csv(path, sep='	', header=None, names=['text','labels','id'], quoting=3, engine='python')

def encode_labels_str(s, num_classes=NUM_EMOTIONS):
    v = np.zeros(num_classes, dtype=np.float32)
    if pd.isna(s):
        return v
    for p in re.split(r'[\s,]+', str(s).strip()):
        if p.isdigit():
            idx = int(p)
            if 0 <= idx < num_classes: v[idx] = 1.0
    return v

train_df = read_tsv(TRAIN_PATH)
dev_df = read_tsv(DEV_PATH)
print('Loaded:', len(train_df), len(dev_df))
train_df['label_vec'] = train_df['labels'].apply(lambda s: encode_labels_str(s, NUM_EMOTIONS))
dev_df['label_vec'] = dev_df['labels'].apply(lambda s: encode_labels_str(s, NUM_EMOTIONS))
label_matrix = np.stack(train_df['label_vec'].values)
class_counts = label_matrix.sum(axis=0).astype(int)
print('Top 8 classes:', np.argsort(-class_counts)[:8].tolist())
print('Class counts (sample):', [(i,int(c)) for i,c in enumerate(class_counts) if c>0][:10])


In [ ]:
# compute pos_weights (neg/pos) for stable BCEWithLogitsLoss
total_samples = len(train_df)
pos_weights = []
for c in range(NUM_EMOTIONS):
    pos = max(1, int(class_counts[c]))
    neg = max(1, total_samples - pos)
    pos_weights.append(float(neg / pos))
pos_weights = np.array(pos_weights, dtype=np.float32)
print('pos_weights sample:', pos_weights[:8])

# Tokenizer + Dataset
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        item = {k: v.squeeze(0) for k,v in enc.items()}
        item['labels'] = torch.as_tensor(self.labels[idx], dtype=torch.float32)
        return item

train_dataset = EmotionDataset(train_df['text'].tolist(), train_df['label_vec'].tolist(), tokenizer)
dev_dataset = EmotionDataset(dev_df['text'].tolist(), dev_df['label_vec'].tolist(), tokenizer)
# per-sample weight: mean pos_weight of active classes (fallback 1.0)
pw = pos_weights.tolist()
sample_weights = []
for v in train_df['label_vec'].values:
    active = np.where(v>0)[0]
    if len(active)==0:
        sample_weights.append(1.0)
    else:
        sample_weights.append(float(np.mean([pw[i] for i in active])))
sample_weights = np.array(sample_weights, dtype=np.float32)
sample_weights = sample_weights / sample_weights.mean()
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
dev_loader = DataLoader(dev_dataset, batch_size=EVAL_BATCH, shuffle=False, num_workers=2)
print('Data loaders:', len(train_loader), 'train batches;', len(dev_loader), 'dev batches')


In [ ]:
# Model, loss, optimizer, scheduler
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_EMOTIONS, problem_type='multi_label_classification')
model.to(DEVICE)
pos_weight_t = torch.tensor(pos_weights, dtype=torch.float32).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
# param groups
head_params = [p for n,p in model.named_parameters() if 'classifier' in n or 'pooler' in n]
backbone_params = [p for n,p in model.named_parameters() if not ('classifier' in n or 'pooler' in n)]
optimizer = AdamW([{'params': backbone_params, 'lr': BACKBONE_LR},{'params': head_params, 'lr': HEAD_LR}], weight_decay=WEIGHT_DECAY)
total_steps = max(1, len(train_loader) * NUM_EPOCHS)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP, num_training_steps=total_steps)
print('Model ready on', DEVICE)
# Setup logging
writer = SummaryWriter(log_dir=str(TB_DIR))
metrics_csv = METRICS_DIR / 'training_metrics.csv'
with open(metrics_csv, 'w') as f:
    f.write('epoch,train_loss,val_loss,micro_f1,macro_f1\n')
step_csv = METRICS_DIR / 'training_step_metrics.csv'
with open(step_csv, 'w') as f:
    f.write('global_step,epoch,step,loss,precision,recall,f1,lr,grad_norm\n')


In [ ]:
def grad_norm(model):
    total = 0.0
    count = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2).item()
            total += param_norm ** 2
            count += 1
    return math.sqrt(total) if count>0 else 0.0

def get_lr(optimizer):
    return optimizer.param_groups[0]['lr']


In [ ]:
# Training loop with detailed logging (per-step metrics every LOG_BATCH steps)
best_val = float('inf')
best_micro = 0.0
patience = 0
LOG_BATCH = 20
global_step = 0
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    step = 0
    for batch in tqdm(train_loader, desc=f'Train E{epoch+1}'):
        labels = batch.pop('labels').to(DEVICE)
        for k in list(batch.keys()): batch[k] = batch[k].to(DEVICE)
        optimizer.zero_grad()
        out = model(**batch)
        logits = out.logits
        if torch.isnan(logits).any() or torch.isinf(logits).any():
            torch.save({'epoch':epoch,'step':step,'logits':logits.detach().cpu()}, CHECKPOINT_DIR / f'nan_logits_ep{epoch}_st{step}.pt')
            raise RuntimeError('NaN in logits')
        loss = loss_fn(logits, labels)
        if torch.isnan(loss) or torch.isinf(loss):
            torch.save({'epoch':epoch,'step':step,'labels':labels.detach().cpu(),'logits':logits.detach().cpu()}, CHECKPOINT_DIR / f'nan_loss_ep{epoch}_st{step}.pt')
            raise RuntimeError('NaN loss')
        loss.backward()
        gnorm = grad_norm(model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        # per-step logging every LOG_BATCH steps
        if global_step % LOG_BATCH == 0:
            # compute batch-level precision/recall/f1 (micro) on this minibatch
            with torch.no_grad():
                probs = torch.sigmoid(logits).cpu().numpy()
                preds = (probs >= 0.5).astype(int)
                y_true = labels.cpu().numpy()
                try:
                    prec, rec, f1, _ = precision_recall_fscore_support(y_true, preds, average='micro', zero_division=0)
                except Exception:
                    prec = rec = f1 = 0.0
            cur_lr = get_lr(optimizer)
            writer.add_scalar('train/step_loss', loss.item(), global_step)
            writer.add_scalar('train/step_prec', prec, global_step)
            writer.add_scalar('train/step_rec', rec, global_step)
            writer.add_scalar('train/step_f1', f1, global_step)
            writer.add_scalar('train/step_grad_norm', gnorm, global_step)
            writer.add_scalar('train/step_lr', cur_lr, global_step)
            # append to step CSV
            with open(step_csv, 'a') as sf:
                sf.write(f'{global_step},{epoch+1},{step},{loss.item():.6f},{prec:.6f},{rec:.6f},{f1:.6f},{cur_lr:.6e},{gnorm:.6f}\n')
        global_step += 1
        step += 1
    avg_train = running_loss / max(1, len(train_loader))
    # Validation
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(dev_loader, desc='Valid'):
            labels = batch.pop('labels').to(DEVICE)
            for k in list(batch.keys()): batch[k] = batch[k].to(DEVICE)
            out = model(**batch)
            logits = out.logits
            loss = loss_fn(logits, labels)
            val_loss += loss.item()
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())
    avg_val = val_loss / max(1, len(dev_loader))
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    micro = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    print(f'Epoch {epoch+1}: TrainLoss={avg_train:.4f} ValLoss={avg_val:.4f} MicroF1={micro:.4f} MacroF1={macro:.4f}')
    # logging
    writer.add_scalar('epoch/train_loss', avg_train, epoch+1)
    writer.add_scalar('epoch/val_loss', avg_val, epoch+1)
    writer.add_scalar('epoch/micro_f1', micro, epoch+1)
    writer.add_scalar('epoch/macro_f1', macro, epoch+1)
    with open(metrics_csv, 'a') as f:
        f.write(f'{epoch+1},{avg_train:.6f},{avg_val:.6f},{micro:.6f},{macro:.6f}\n')
    # save sample predictions and probabilities for inspection
    np.save(METRICS_DIR / f'val_preds_ep{epoch+1}.npy', all_preds)
    np.save(METRICS_DIR / f'val_labels_ep{epoch+1}.npy', all_labels)
    # checkpointing by micro F1 and val loss
    if micro > best_micro or avg_val < best_val:
        best_micro = max(best_micro, micro)
        best_val = min(best_val, avg_val)
        model.save_pretrained(CHECKPOINT_DIR / f'best_epoch_{epoch+1}')
        print('Saved best model at epoch', epoch+1)
        patience = 0
    else:
        patience += 1
        if patience >= 2:
            print('Early stopping')
            break
writer.flush()
print('Training complete')


In [ ]:
# Final evaluation on dev using threshold sweep and save report
best_ckpts = sorted([p for p in CHECKPOINT_DIR.iterdir() if p.is_dir()], key=lambda d: d.stat().st_mtime)
if best_ckpts:
    print('Loading', best_ckpts[-1])
    model = AutoModelForSequenceClassification.from_pretrained(str(best_ckpts[-1]))
    model.to(DEVICE)
model.eval()
probs_list, labels_list = [], []
with torch.no_grad():
    for batch in dev_loader:
        labels = batch.pop('labels')
        for k in list(batch.keys()): batch[k] = batch[k].to(DEVICE)
        out = model(**batch)
        probs = torch.sigmoid(out.logits).cpu().numpy()
        probs_list.append(probs)
        labels_list.append(labels.numpy())
probs = np.vstack(probs_list)
labels = np.vstack(labels_list)
# threshold sweep per class (coarse)
best_thresh = np.zeros(NUM_EMOTIONS)
for i in range(NUM_EMOTIONS):
    best_f1, best_t = 0.0, 0.5
    for t in np.linspace(0.1, 0.9, 17):
        p = (probs[:,i] >= t).astype(int)
        f1 = f1_score(labels[:,i], p, zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, t
    best_thresh[i] = best_t
print('thresholds sample:', best_thresh[:6])
preds_opt = (probs >= best_thresh).astype(int)
for r in range(preds_opt.shape[0]):
    if preds_opt[r].sum() == 0: preds_opt[r, probs[r].argmax()] = 1
micro = f1_score(labels, preds_opt, average='micro', zero_division=0)
macro = f1_score(labels, preds_opt, average='macro', zero_division=0)
print(f'Final Dev MicroF1={micro:.4f} MacroF1={macro:.4f}')
report = classification_report(labels, preds_opt, output_dict=True, zero_division=0)
with open(METRICS_DIR / 'final_report.json', 'w') as f: json.dump({'micro':micro,'macro':macro,'report':report,'thresholds': best_thresh.tolist()}, f, indent=2)
print('Saved final_report.json')
